In [2]:
pip install pandas openpyxl xlrd

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import re
import numpy as np

df = pd.read_excel("BOQ_1833226.xls", skiprows=10)


description_column = next((col for col in df.columns if isinstance(col, str) and "description" in col.lower()), None)
qty_column = next((col for col in df.columns if isinstance(col, str) and "qty" in col.lower()), None)
unit_column = next((col for col in df.columns if isinstance(col, str) and "unit" in col.lower()), None)
rate_column = next((col for col in df.columns if isinstance(col, str) and "rate" in col.lower()), None)
amount_column = next((col for col in df.columns if isinstance(col, str) and "amount" in col.lower()), None)

if not description_column:
    raise Exception("Could not find 'Item Description' column.")

class_pattern = r"\b(DI|HDPE|PVC|UPVC|CI|MS)\b"
dia_pattern = r"\b(\d{2,4})\s*(mm|MM)?\b"

name_of_work = str(df.iloc[0][description_column])[:80]

results = []

for _, row in df.iterrows():
    text = str(row.get(description_column, "")).strip()

    if len(text) < 5 or text.lower() in ["nan", ""]:
        continue

    class_match = re.findall(class_pattern, text)
    dia_match = re.findall(dia_pattern, text)

    dia_values = [m[0] for m in dia_match]
    class_values = list(set(class_match))

    def safe_number(val):
        try:
            return float(val) if pd.notna(val) else None
        except:
            return None

    unit_val = str(row.get(unit_column, "")).strip() if unit_column else ""
    rate_val = safe_number(row.get(rate_column, "")) if rate_column else None
    amount_val = safe_number(row.get(amount_column, "")) if amount_column else None
    qty_val = safe_number(row.get(qty_column, "")) if qty_column else None

    if qty_val in [None, "", 0] and rate_val and amount_val:
        try:
            qty_val = round(amount_val / rate_val, 2)
        except ZeroDivisionError:
            qty_val = ""

    result = {
        "tender_details_id": "BOQ_1833226",
        "name_of_work": name_of_work,
        "item_description": text[:150],
        "class": ",".join(class_values),
        "DIA": ",".join(set(dia_values)),
        "unit": unit_val,
        "quantity": qty_val,
        "estimate rate": rate_val,
        "total amount with taxes": amount_val
    }

    if result["class"] or result["DIA"] or result["quantity"]:
        results.append(result)
final_df = pd.DataFrame(results)
final_df.replace({np.nan: ""}, inplace=True)

output_path = "boq_final_with_quantity.xlsx"
final_df.to_excel(output_path, index=False)

print(f" File saved: {output_path}")
final_df.head()

 File saved: boq_final_with_quantity.xlsx


,tender_details_id,name_of_work,item_description,class,DIA,unit,quantity,estimate rate,total amount with taxes
0,BOQ_1833226,2,Cost for DI K-9 & M.S Pipes with Fittings & Va...,DI,,nan,,,
1,BOQ_1833226,2,Carrying CI/DI pipes with specials and valves ...,"DI,CI",150,Mtr.,4905.0,112.0,549360.0
2,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3420.0,136.0,465120.0
3,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2040.0,164.0,334560.0
4,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,267.5,192.0,51360.0


In [4]:
pd.set_option("display.max_rows", None)  

In [5]:
final_df.head()

,tender_details_id,name_of_work,item_description,class,DIA,unit,quantity,estimate rate,total amount with taxes
0,BOQ_1833226,2,Cost for DI K-9 & M.S Pipes with Fittings & Va...,DI,,nan,,,
1,BOQ_1833226,2,Carrying CI/DI pipes with specials and valves ...,"DI,CI",150,Mtr.,4905.0,112.0,549360.0
2,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3420.0,136.0,465120.0
3,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2040.0,164.0,334560.0
4,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,267.5,192.0,51360.0


In [6]:
final_df

,tender_details_id,name_of_work,item_description,class,DIA,unit,quantity,estimate rate,total amount with taxes
0,BOQ_1833226,2,Cost for DI K-9 & M.S Pipes with Fittings & Va...,DI,,nan,,,
1,BOQ_1833226,2,Carrying CI/DI pipes with specials and valves ...,"DI,CI",150,Mtr.,4905.0,112.0,549360.0
2,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3420.0,136.0,465120.0
3,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2040.0,164.0,334560.0
4,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,267.5,192.0,51360.0
5,BOQ_1833226,2,"Filling the pipe line with water, testing hydr...",,"150,10,00,3114,1965",Mtr.,4955.0,4.2,20811.0
6,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3455.0,5.61,19382.55
7,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2060.0,6.66,13719.6
8,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,270.0,7.22,1949.4
9,BOQ_1833226,2,Cleaning thoroughly the inner surface of pipe ...,,"150,10",Mtr.,4955.0,1.54,7630.7


In [7]:
output_path = "boq_final_with_quantity.xlsx"
final_df.to_excel(output_path, index=False)

In [8]:
final_df

,tender_details_id,name_of_work,item_description,class,DIA,unit,quantity,estimate rate,total amount with taxes
0,BOQ_1833226,2,Cost for DI K-9 & M.S Pipes with Fittings & Va...,DI,,nan,,,
1,BOQ_1833226,2,Carrying CI/DI pipes with specials and valves ...,"DI,CI",150,Mtr.,4905.0,112.0,549360.0
2,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3420.0,136.0,465120.0
3,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2040.0,164.0,334560.0
4,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,267.5,192.0,51360.0
5,BOQ_1833226,2,"Filling the pipe line with water, testing hydr...",,"150,10,00,3114,1965",Mtr.,4955.0,4.2,20811.0
6,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3455.0,5.61,19382.55
7,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2060.0,6.66,13719.6
8,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,270.0,7.22,1949.4
9,BOQ_1833226,2,Cleaning thoroughly the inner surface of pipe ...,,"150,10",Mtr.,4955.0,1.54,7630.7


In [9]:
def is_valid_dia(val):
    if pd.isna(val):
        return True
    parts = str(val).split(',')
    return all(p.strip().isdigit() for p in parts) and len(parts) <= 3

final_df = final_df[final_df["DIA"].apply(is_valid_dia)]

In [10]:
unwanted_units = ["Each", "Kg", "Cum","sqm","MT","LS"]
final_df = final_df[final_df["unit"].notna()]
final_df = final_df[~final_df["unit"].isin(unwanted_units)]

final_df.reset_index(drop=True, inplace=True)

In [11]:
final_df

,tender_details_id,name_of_work,item_description,class,DIA,unit,quantity,estimate rate,total amount with taxes
0,BOQ_1833226,2,Carrying CI/DI pipes with specials and valves ...,"DI,CI",150,Mtr.,4905.0,112.0,549360.0
1,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3420.0,136.0,465120.0
2,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2040.0,164.0,334560.0
3,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,267.5,192.0,51360.0
4,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3455.0,5.61,19382.55
5,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2060.0,6.66,13719.6
6,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,270.0,7.22,1949.4
7,BOQ_1833226,2,Cleaning thoroughly the inner surface of pipe ...,,"150,10",Mtr.,4955.0,1.54,7630.7
8,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3455.0,2.38,8222.9
9,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2060.0,3.47,7148.2


In [12]:
final_df = final_df[final_df["estimate rate"] >= 100]

final_df.reset_index(drop=True, inplace=True)

In [13]:
final_df

,tender_details_id,name_of_work,item_description,class,DIA,unit,quantity,estimate rate,total amount with taxes
0,BOQ_1833226,2,Carrying CI/DI pipes with specials and valves ...,"DI,CI",150,Mtr.,4905.0,112.0,549360.0
1,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3420.0,136.0,465120.0
2,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,2040.0,164.0,334560.0
3,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,267.5,192.0,51360.0
4,BOQ_1833226,2,Making barricading of excavation trenches of r...,,75,Mtr.,107.4,118.0,12673.2
5,BOQ_1833226,2,Laying Cost of M.S pipe \n150 mm dia Pipe,,150,Mtr.,50.0,564.38,28219.0
6,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,35.0,929.52,32533.2
7,BOQ_1833226,2,250 mm dia Pipe,,250,Mtr.,20.0,1118.23,22364.6
8,BOQ_1833226,2,300 mm dia Pipe,,300,Mtr.,2.5,1755.08,4387.7
9,BOQ_1833226,2,200 mm dia Pipe,,200,Mtr.,3420.0,2881.0,9853020.0


In [14]:
final_df = final_df.drop(columns=['name_of_work'])

In [15]:
final_df

,tender_details_id,item_description,class,DIA,unit,quantity,estimate rate,total amount with taxes
0,BOQ_1833226,Carrying CI/DI pipes with specials and valves ...,"DI,CI",150,Mtr.,4905.0,112.0,549360.0
1,BOQ_1833226,200 mm dia Pipe,,200,Mtr.,3420.0,136.0,465120.0
2,BOQ_1833226,250 mm dia Pipe,,250,Mtr.,2040.0,164.0,334560.0
3,BOQ_1833226,300 mm dia Pipe,,300,Mtr.,267.5,192.0,51360.0
4,BOQ_1833226,Making barricading of excavation trenches of r...,,75,Mtr.,107.4,118.0,12673.2
5,BOQ_1833226,Laying Cost of M.S pipe \n150 mm dia Pipe,,150,Mtr.,50.0,564.38,28219.0
6,BOQ_1833226,200 mm dia Pipe,,200,Mtr.,35.0,929.52,32533.2
7,BOQ_1833226,250 mm dia Pipe,,250,Mtr.,20.0,1118.23,22364.6
8,BOQ_1833226,300 mm dia Pipe,,300,Mtr.,2.5,1755.08,4387.7
9,BOQ_1833226,200 mm dia Pipe,,200,Mtr.,3420.0,2881.0,9853020.0


In [16]:
def classify_pipe_type(text):
    text = text.lower()
    if "ms" in text:
        return "MS Pipe"
    elif "ci" in text or "di" in text:
        return "CI/DI Pipe"
    elif "k9" in text:
        return "K9 Pipe"
    elif "k7" in text:
        return "K7 Pipe"
    elif "supply" in text or "demand" in text:
        return "Supply & Demand"
    else:
        return "Other"

In [17]:
final_df['class'] = final_df['class'].apply(classify_pipe_type)


In [18]:
final_df_simple = final_df[['tender_details_id', 'class', 'DIA', 'estimate rate']]


In [19]:
final_df_simple.to_excel("boq_simplified_output.xlsx", index=False)

In [42]:
final_df_simple.head(10)

,tender_details_id,class,DIA,estimate rate
0,BOQ_1833226,CI/DI Pipe,150,112.0
1,BOQ_1833226,Other,200,136.0
2,BOQ_1833226,Other,250,164.0
3,BOQ_1833226,Other,300,192.0
4,BOQ_1833226,Other,75,118.0
5,BOQ_1833226,Other,150,564.38
6,BOQ_1833226,Other,200,929.52
7,BOQ_1833226,Other,250,1118.23
8,BOQ_1833226,Other,300,1755.08
9,BOQ_1833226,Other,200,2881.0


In [21]:
final_output = final_df[['tender_details_id', 'class', 'DIA', 'unit', 'estimate rate']]


In [41]:
print(final_output.head(20))


   tender_details_id       class     DIA  unit  estimate rate
0        BOQ_1833226  CI/DI Pipe     150  Mtr.         112.00
1        BOQ_1833226       Other     200  Mtr.         136.00
2        BOQ_1833226       Other     250  Mtr.         164.00
3        BOQ_1833226       Other     300  Mtr.         192.00
4        BOQ_1833226       Other      75  Mtr.         118.00
5        BOQ_1833226       Other     150  Mtr.         564.38
6        BOQ_1833226       Other     200  Mtr.         929.52
7        BOQ_1833226       Other     250  Mtr.        1118.23
8        BOQ_1833226       Other     300  Mtr.        1755.08
9        BOQ_1833226       Other     200  Mtr.        2881.00
10       BOQ_1833226       Other     250  Mtr.        3857.00
11       BOQ_1833226       Other     300  Mtr.        4887.00
12       BOQ_1833226       Other  150,18  Mtr.        2505.19
13       BOQ_1833226       Other     200  Mtr.        3812.56
14       BOQ_1833226       Other     250  Mtr.        4398.50
15      

In [25]:
df.columns = df.columns.str.strip()

In [26]:
def detect_pipe_class(desc):
    desc = str(desc).lower()
    if "m.s" in desc or "ms pipe" in desc:
        return "M.S pipe"
    elif "di" in desc or "ci" in desc:
        return "DI,CI"
    elif "k9" in desc:
        return "K9 Pipe"
    elif "k7" in desc:
        return "K7 Pipe"
    elif "supply" in desc and "pipe" in desc:
        return "Supply pipe"
    else:
        return ""

In [30]:
df['class'] = df['Item Description'].apply(detect_pipe_class)

In [32]:
desc_col = next((col for col in df.columns if 'description' in col.lower()), None)

if desc_col:
    df['class'] = df[desc_col].apply(detect_pipe_class)
else:
    raise Exception("Could not find the item description column.")
 

In [37]:
df.head(5)

,Sl.\nNo.,Item Description,Item Code / Make,Quantity,Units,Estimated Rate in \nRs. P,Unnamed: 6,Unnamed: 7,Addition / Deduction,Addition / Deduction Values,...,Unnamed: 234,Unnamed: 235,Unnamed: 236,Unnamed: 237,Unnamed: 238,Unnamed: 239,Unnamed: 240,Unnamed: 241,Unnamed: 242,class
0,1,2,3,4.0,5,6.0,7.0,8.0,9,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
1,1,Cost for DI K-9 & M.S Pipes with Fittings & Va...,BI01010001010000000000000515BI0100001112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.00,Construction of chamber for 100mm sluices valve,item1,10.000,1 Nos,M.S pipe
2,1.01,Carrying CI/DI pipes with specials and valves ...,BI01010001010000000000000515BI0100001113,4905.0,Mtr.,112.0,NaN,NaN,Excess(+),1.0,...,NaN,NaN,NaN,NaN,1.01,"Supplying, Conveying and fixing spls. Includin...",item1,123.223,Nos,"DI,CI"
3,1.02,200 mm dia Pipe,BI01010001010000000000000515BI0100001114,3420.0,Mtr.,136.0,NaN,NaN,Excess(+),1.0,...,NaN,NaN,NaN,NaN,1.02,Construction of chamber for 100mm sluice plates,item2,213.000,Nos,"DI,CI"
4,1.03,250 mm dia Pipe,BI01010001010000000000000515BI0100001115,2040.0,Mtr.,164.0,NaN,NaN,Excess(+),1.0,...,NaN,NaN,NaN,NaN,1.02,Construction of chamber for 100mm sluice plates,item2,213.000,Nos,"DI,CI"
